<a href="https://colab.research.google.com/github/busycaesar/LoRA/blob/Master/google-gemma/gemma-output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Set Up Kaggle Credentials
To download the Gemma model checkpoints, we authenticate using our Kaggle API credentials. If you do not have your credentials yet, you can create and download your API token from your [Kaggle Account](https://www.kaggle.com). Once downloaded, store them securely in Colab's secrets manager.

In [1]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### Step 2: Install Required Libraries

In [2]:
%pip install -U -q keras-hub keras

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.33.0 which is incompatible.


### Step 3: Configure Keras Backend
We are using a tool called **JAX** as our backend because it makes training large AI models run much faster. We also set a memory limit (`XLA_PYTHON_CLIENT_MEM_FRACTION = "0.90"`). This tells JAX to reserve 90% of the graphics card (GPU) memory upfront instead of trying to grab 100%. If we let it take all the memory at once, the notebook might run out of space and crash.

In [3]:
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

### Step 4: Import Frameworks


In [4]:
import keras
import keras_hub

### Step 5: Load Gemma 3

In [5]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")

### Step 6: Define Template
This is the template that we wil use to send the prompt to the LLM. We will apprend the prompt into the `instruction` and leave the `response` empty for the LLM to fill.

In [6]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

### Step 7: Zero-Shot Inference
We test the base model with an initial prompt before any fine-tuning to observe its default performance.

In [7]:
prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)

print(gemma_lm.generate(prompt, max_length=256))

Instruction:
What should I do on a trip to Europe?

Response:
Europe offers a vast array of experiences, and the best trip depends heavily on your interests and budget. Here's a breakdown of potential activities and destinations, categorized by interest:

**1. History & Culture:**

*   **Rome, Italy:** Explore the Colosseum, Roman Forum, and Vatican City.
*   **Athens, Greece:** Visit the Acropolis, ancient temples, and the Parthenon.
*   **Paris, France:** Discover the Louvre Museum, Notre Dame Cathedral, and charming streets.
*   **Berlin, Germany:** Immerse yourself in the history of the Berlin Wall, museums, and WWII sites.
*   **Istanbul, Turkey:** Explore the Hagia Sophia, Blue Mosque, and Topkapi Palace.

**2.  City Exploration & Nightlife:**

*   **Barcelona, Spain:** Enjoy the architecture of Antoni Gaudí, vibrant nightlife, and delicious tapas.
*   **London, England:** Experience the iconic landmarks, theatre, and diverse culture.
*   **Amsterdam, Netherlands:** Explore canal

### Step 8: Download Training Dataset
We fetch the Databricks Dolly 15k dataset, which is a collection of high-quality, human-generated instruction/response pairs.

In [8]:
!wget -O databricks-dolly-15k.jsonl https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

--2026-11-14 22:46:14--  https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl
Resolving huggingface.co (huggingface.co)... 99.86.101.56, 99.86.101.64, 99.86.101.39, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.56|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64358e2179c45fcf1ada09f4/63c4dabe683d7254493568d2d3995c0e51abc8528ef3b4936497c538cb501e93?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27databricks-dolly-15k.jsonl%3B+filename%3D%22databricks-dolly-15k.jsonl%22%3B&X-Xet-Cas-Uid=public&user_id=public&xip=_wlRIW9fNKM&Expires=1794699974&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjQzNThlMjE3OWM0NWZjZjFhZGEwOWY0LzYzYzRkYWJlNjgzZDcyNTQ0OTM1NjhkMmQzOTk1YzBlNTFhYmM4NTI4ZWYzYjQ5MzY0OTdjNTM4Y2I1MDFlOTNcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMmdXNlcl9pZD1wdWJs

### Step 9: Parse and Format Dataset
We parse the first 1000 examples, filtering out items that require background context to keep our instruction-response fine-tuning setup simple.

In [9]:
import json

prompts = []
responses = []
line_count = 0

with open("databricks-dolly-15k.jsonl") as file:
    for line in file:
        if line_count >= 1000:
            break  # Limit the training examples, to reduce execution time.

        examples = json.loads(line)
        # Filter out examples with context, to keep it simple.
        if examples["context"]:
            continue
        # Format data into prompts and response lists.
        prompts.append(examples["instruction"])
        responses.append(examples["response"])

        line_count += 1

features = {
    "prompts": prompts,
    "responses": responses
}

### Step 10: Verify Parsed Data
We print out the dataset size and sample prompt-response pairs to verify our data processing step worked correctly.

In [10]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

1000
Which is a species of fish? Tope or Rope
Tope


## Summary of the LLM

In [11]:
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

### Step 11: Enable Low-Rank Adaptation (LoRA)
By calling `.enable_lora(rank=4)` on the backbone, we significantly reduce the number of trainable parameters, making fine-tuning highly efficient on standard GPU hardware.

In [12]:
gemma_lm.backbone.enable_lora(rank=4)

In [13]:
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │   1,000,538,240 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,000,538,240 (3.73 GB)

 Trainable params: 652,288 (2.49 MB)

 Non-trainable params: 999,885,952 (3.72 GB)

### Step 12: Compile Model
We prepare the model for training by applying standard settings (an optimizer and a loss function) that will help it learn from our dataset.

In [14]:
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)

optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

### Step 13: Model Training

In [15]:
gemma_lm.fit(features, epochs=1, batch_size=1)

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a grain dataset. The grain dataset is expected to already be shuffled.
  self.data_adapter = data_adapters.get_data_adapter(


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1170s 1s/step - loss: 0.2108 - sparse_categorical_accuracy: 0.4872


### Step 14: Evaluate Fine-Tuned Model

In [16]:
prompt = template.format(
    instruction="What should I do on a trip to Europe?",
    response="",
)

print(gemma_lm.generate(prompt, max_length=256))

Instruction:
What should I do on a trip to Europe?

Response:
Europe is a great place to visit. There are many things to do, such as visit museums, go to the beach, or go hiking. There are also many restaurants to eat.
